# Robust Evaluation – 10 Stratified 80/20 Splits

In this notebook we perform a fair evaluation of the Random Forest models using the 
hyperparameters previously obtained via Bayesian Optimization.

Instead of relying on a single train/test split, we perform:

- 10 independent 80/20 stratified splits
- Retrain models from scratch on each split
- Evaluate on each corresponding test set
- Average performance metrics across splits

This removes the effect of a potentially favorable split and provides a robust estimate of performance.

We evaluate both approaches:

1. Independent Binary classifiers
2. Label Power Set (LPS) classifier

Metrics computed 
- Accuracy (ACC)
- Hamming Loss (HL)
- Weighted F1 (WF1)

For LPS:
Predicted patterns are decomposed into individual binary labels before computing metrics.

# Imports and Configuration

In [6]:
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, hamming_loss, f1_score
import time
from tqdm import tqdm

RANDOM_SEEDS = list(range(10))
TEST_SIZE = 0.20

## Utility Functions

In [2]:
def multilabel_weighted_f1(y_true, y_pred):
    total = 0
    for col in range(y_true.shape[1]):
        total += f1_score(y_true[:, col], y_pred[:, col], average="weighted")
    return total / y_true.shape[1]

## Load dataset

In [3]:
with open("../data/DRIAMS_A_AMR_paper_replication.pkl", "rb") as f:
    payload = pickle.load(f)

X_raw = payload["data"]
amr_raw = payload["amr"]
antibiotics = payload["antibiotics"]
labels_raw = payload["label"]

df_features = pd.DataFrame(X_raw)
df_amr = pd.DataFrame(amr_raw, columns=antibiotics)
df_species = pd.DataFrame(labels_raw, columns=["species"])

full_df = pd.concat([df_features, df_amr, df_species], axis=1)
full_df = full_df.drop_duplicates()

n_features = X_raw.shape[1]
feature_cols = list(full_df.columns[:n_features])

## Hyperparameters (From Bayesian Optimization)

In [4]:
hyperparams_binary = {
    "Staphylococcus_Aureus": {
        "Oxacillin": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 42},
        "Clindamycin": {'bootstrap': True, 'max_depth': 5, 'min_samples_leaf': 9, 'n_estimators': 1},
        "Fusidic acid": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 447},
    },
    "Escherichia_Coli": {
        "Ciprofloxacin": {'bootstrap': False, 'max_depth': 9, 'min_samples_leaf': 1, 'n_estimators': 1},
        "Ceftriaxone": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 6},
        "Piperacillin-Tazobactam": {'bootstrap': False, 'max_depth': 5, 'min_samples_leaf': 2, 'n_estimators': 334},
        "Cefepime": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 5},
    },
    "Klebsiella_Pneumoniae": {
        "Ciprofloxacin": {'bootstrap': False, 'max_depth': 7, 'min_samples_leaf': 10, 'n_estimators': 3},
        "Ceftriaxone": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 17},
        "Imipenem": {'bootstrap': True, 'max_depth': 7, 'min_samples_leaf': 7, 'n_estimators': 647},
        "Meropenem": {'bootstrap': True, 'max_depth': 7, 'min_samples_leaf': 7, 'n_estimators': 647},
    },
    "Pseudomonas_Aeruginosa": {
        "Ciprofloxacin": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 520},
        "Imipenem": {'bootstrap': False, 'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 2},
        "Meropenem": {'bootstrap': True, 'max_depth': 7, 'min_samples_leaf': 1, 'n_estimators': 9},
    }
}

hyperparams_lps = {
    "Staphylococcus_Aureus": {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 2},
    "Escherichia_Coli": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 2},
    "Klebsiella_Pneumoniae": {'bootstrap': False, 'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 4},
    "Pseudomonas_Aeruginosa": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 341},
}

## Evaluation Loop – 10 Stratified Splits

For each species:
- Remove rare patterns (<10 samples)
- Perform 10 independent 80/20 splits (stratified by pattern)
- Train binary models
- Train LPS model
- Decompose LPS predictions into multi-label
- Compute ACC, HL, WF1
- Average across splits

In [ ]:
species_antibiotics = {
    "Staphylococcus_Aureus": ["Oxacillin", "Clindamycin", "Fusidic acid"],
    "Escherichia_Coli": ["Ciprofloxacin", "Ceftriaxone", "Piperacillin-Tazobactam", "Cefepime"],
    "Klebsiella_Pneumoniae": ["Ciprofloxacin", "Ceftriaxone", "Imipenem", "Meropenem"],
    "Pseudomonas_Aeruginosa": ["Ciprofloxacin", "Imipenem", "Meropenem"],
}

results = {}

global_start = time.time()

for species, ab_list in species_antibiotics.items():

    print("\n" + "="*70, flush=True)
    print(f"Processing species: {species}", flush=True)
    print("="*70, flush=True)

    species_start = time.time()

    df_sp = full_df[full_df["species"] == species]
    print(f"Initial samples: {df_sp.shape[0]}", flush=True)

    sp_df = df_sp[feature_cols + ab_list].dropna()
    print(f"After NaN removal: {sp_df.shape[0]}", flush=True)

    X = sp_df.iloc[:, :n_features].to_numpy()
    y_multi = sp_df[ab_list].to_numpy().astype(int)

    patterns = np.array(["".join(map(str, row)) for row in y_multi])
    counts = pd.Series(patterns).value_counts()
    valid = counts[counts >= 10].index
    mask = np.isin(patterns, valid)

    X = X[mask]
    y_multi = y_multi[mask]
    patterns = patterns[mask]

    print(f"After rare-pattern filtering (<10 removed): {X.shape[0]}", flush=True)
    print(f"Unique patterns kept: {len(np.unique(patterns))}", flush=True)

    metrics_bin = []
    metrics_lps = []

    for seed in tqdm(RANDOM_SEEDS, desc=f"{species} splits"):

        split_start = time.time()

        train_idx, test_idx = train_test_split(
            np.arange(X.shape[0]),
            test_size=TEST_SIZE,
            random_state=seed,
            stratify=patterns
        )

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_multi[train_idx], y_multi[test_idx]
        patterns_train = patterns[train_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # ===============================
        # Binary Models
        # ===============================
        preds_bin = []

        for i, ab in enumerate(ab_list):
            params = hyperparams_binary[species][ab]
            model = RandomForestClassifier(**params, random_state=0)
            model.fit(X_train, y_train[:, i])
            preds_bin.append(model.predict(X_test))

        preds_bin = np.array(preds_bin).T

        acc_bin = accuracy_score(y_test, preds_bin)
        ham_bin = hamming_loss(y_test, preds_bin)
        f1_bin = multilabel_weighted_f1(y_test, preds_bin)

        metrics_bin.append((acc_bin, ham_bin, f1_bin))

        # ===============================
        # LPS Model
        # ===============================
        le = LabelEncoder()
        y_train_lps = le.fit_transform(patterns_train)

        model_lps = RandomForestClassifier(
            **hyperparams_lps[species],
            random_state=0
        )
        model_lps.fit(X_train, y_train_lps)

        pred_lps = model_lps.predict(X_test)
        decoded = le.inverse_transform(pred_lps)
        preds_multi = np.array([[int(c) for c in s] for s in decoded])

        acc_lps = accuracy_score(y_test, preds_multi)
        ham_lps = hamming_loss(y_test, preds_multi)
        f1_lps = multilabel_weighted_f1(y_test, preds_multi)

        metrics_lps.append((acc_lps, ham_lps, f1_lps))

        split_time = time.time() - split_start
        print(f"Seed {seed} done in {split_time:.2f} sec", flush=True)

    results[species] = {
        "binary_mean": np.mean(metrics_bin, axis=0),
        "binary_std": np.std(metrics_bin, axis=0),
        "lps_mean": np.mean(metrics_lps, axis=0),
        "lps_std": np.std(metrics_lps, axis=0),
    }

    species_time = time.time() - species_start
    print(f"\nFinished {species} in {species_time/60:.2f} minutes", flush=True)

total_time = time.time() - global_start
print("\n" + "="*70)
print(f"TOTAL EXECUTION TIME: {total_time/60:.2f} minutes")
print("="*70)

results
results


Processing species: Staphylococcus_Aureus
Initial samples: 3791
After NaN removal: 3556
After rare-pattern filtering (<10 removed): 3556
Unique patterns kept: 8


Staphylococcus_Aureus splits:   0%|          | 0/10 [00:00<?, ?it/s]

Seed 0 done in 49.67 sec


Staphylococcus_Aureus splits:  10%|█         | 1/10 [00:49<07:27, 49.67s/it]

Seed 1 done in 49.45 sec


Staphylococcus_Aureus splits:  20%|██        | 2/10 [01:39<06:36, 49.54s/it]